# 作业 04　正则表达式与结构化网页数据

姓名：欧阳洋　　学号：20244080124

In [1]:
import hashlib
import os
import platform
import uuid

STUDENT_ID = 20244080124

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")

[Conda 环境] python_work1
[学号]       20244080124
[机器指纹] 66ab728912afd30e
[RUN_ID]   20244080124-66ab728912afd30e


## 一、Python 正则表达式练习

每题先定义函数，再用 `assert` 比较实际结果和预期结果。`repr` 在输出中保留空格、空串和换行等细节。

In [2]:
import re


def check_cases(function, cases):
    for value, expected in cases:
        actual = function(value)
        assert actual == expected, (function.__name__, value, expected, actual)
        print(f"{value!r} → {actual!r}")

### 1. 检查仅包含英文字母和数字的非空字符串

In [3]:
def is_ascii_alphanumeric(text):
    return re.fullmatch(r"[a-zA-Z0-9]+", text) is not None


check_cases(is_ascii_alphanumeric, [
    ("AbZ019", True), ("abc-123", False), ("中文42", False), ("A１", False), ("", False),
])

'AbZ019' → True
'abc-123' → False
'中文42' → False
'A１' → False
'' → False


### 2. 匹配一个 `a` 后跟零个或多个 `b`

In [4]:
def matches_a_with_optional_bs(text):
    return re.fullmatch(r"ab*", text) is not None


check_cases(matches_a_with_optional_bs, [
    ("a", True), ("abbb", True), ("aba", False), ("b", False), ("", False),
])

'a' → True
'abbb' → True
'aba' → False
'b' → False
'' → False


### 3. 匹配一个 `a` 后跟一个或多个 `b`

In [5]:
def matches_a_with_required_bs(text):
    return re.fullmatch(r"ab+", text) is not None


check_cases(matches_a_with_required_bs, [
    ("ab", True), ("abbb", True), ("a", False), ("abbc", False), ("aab", False),
])

'ab' → True
'abbb' → True
'a' → False
'abbc' → False
'aab' → False


### 4. 匹配以 `a` 开头、以 `b` 结尾的字符串

In [6]:
def starts_a_ends_b(text):
    return re.fullmatch(r"a.*b", text, flags=re.DOTALL) is not None


check_cases(starts_a_ends_b, [
    ("ab", True), ("a !汉\nb", True), ("aba", False), ("xab", False), ("a", False),
])

'ab' → True
'a !汉\nb' → True
'aba' → False
'xab' → False
'a' → False


### 5. 删除 IPv4 地址各段的前导零

In [7]:
def strip_ip_leading_zeros(address):
    # 每段至少保留最后一位数字，因此全零段输出为 0。
    return re.sub(r"(^|\.)0+(?=[0-9])", r"\1", address)


check_cases(strip_ip_leading_zeros, [
    ("192.168.001.010", "192.168.1.10"),
    ("000.000.00.0", "0.0.0.0"),
    ("001.20.003.200", "1.20.3.200"),
    ("192.168.1.10", "192.168.1.10"),
])

'192.168.001.010' → '192.168.1.10'
'000.000.00.0' → '0.0.0.0'
'001.20.003.200' → '1.20.3.200'
'192.168.1.10' → '192.168.1.10'


### 6. 互换空格与下划线

In [8]:
def swap_spaces_and_underscores(text):
    return re.sub(r"[ _]", lambda match: "_" if match[0] == " " else " ", text)


check_cases(swap_spaces_and_underscores, [
    ("a b_c", "a_b c"), (" _  __", "_ __  "), ("a\t_b", "a\t b"), ("", ""),
])

'a b_c' → 'a_b c'
' _  __' → '_ __  '
'a\t_b' → 'a\t b'
'' → ''


### 7. 将 `yyyy-mm-dd` 转为 `dd-mm-yyyy`

In [9]:
def reorder_date(text):
    return re.sub(r"\b([0-9]{4})-([0-9]{2})-([0-9]{2})\b", r"\3-\2-\1", text)


# 只重排格式，输入可包含一个或多个日期。
check_cases(reorder_date, [
    ("2024-02-29", "29-02-2024"),
    ("From 2026-01-05 to 2026-11-25.", "From 05-01-2026 to 25-11-2026."),
    ("2026-1-5", "2026-1-5"),
])

'2024-02-29' → '29-02-2024'
'From 2026-01-05 to 2026-11-25.' → 'From 05-01-2026 to 25-11-2026.'
'2026-1-5' → '2026-1-5'


### 8. 找出以 `a` 或 `e` 开头的英文单词

In [10]:
def find_a_or_e_words(text):
    return re.findall(r"\b[aAeE][A-Za-z]*\b", text)


check_cases(find_a_or_e_words, [
    ("An eagle and a bear ate eggs.", ["An", "eagle", "and", "a", "ate", "eggs"]),
    ("Elephant, apple; peach.", ["Elephant", "apple"]),
    ("tea, bread, y3ear", []),
])

'An eagle and a bear ate eggs.' → ['An', 'eagle', 'and', 'a', 'ate', 'eggs']
'Elephant, apple; peach.' → ['Elephant', 'apple']
'tea, bread, y3ear' → []


### 9. 将完整单词 `Road` 缩写为 `Rd`

In [11]:
def abbreviate_road(text):
    return re.sub(r"\bRoad\b", "Rd", text)


check_cases(abbreviate_road, [
    ("Main Road, Lake Road.", "Main Rd, Lake Rd."),
    ("Roadside, Broadway, road", "Roadside, Broadway, road"),
    ("Road", "Rd"),
])

'Main Road, Lake Road.' → 'Main Rd, Lake Rd.'
'Roadside, Broadway, road' → 'Roadside, Broadway, road'
'Road' → 'Rd'


### 10. 找出长度至少为 4 的英文单词

In [12]:
def find_long_words(text):
    return re.findall(r"\b[A-Za-z]{4,}\b", text)


check_cases(find_long_words, [
    ("A sun rises over the hill.", ["rises", "over", "hill"]),
    ("cat dog four", ["four"]),
    ("abc 1234 id_123", []),
])

'A sun rises over the hill.' → ['rises', 'over', 'hill']
'cat dog four' → ['four']
'abc 1234 id_123' → []


### 11. 删除除字母和数字以外的字符

In [13]:
def keep_letters_and_numbers(text):
    return re.sub(r"[\W_]+", "", text)


check_cases(keep_letters_and_numbers, [
    ("a_B-3 !?", "aB3"), ("Été42", "Été42"), ("_ \t!🙂", ""), ("X9", "X9"),
])

'a_B-3 !?' → 'aB3'
'Été42' → 'Été42'
'_ \t!🙂' → ''
'X9' → 'X9'


### 12. 删除圆括号与括号内的内容

In [14]:
def remove_parenthetical(text):
    # 每轮先移除最内层的成对圆括号，直到没有匹配项。
    while True:
        text, removed = re.subn(r"\([^()]*\)", "", text)
        if removed == 0:
            return text.strip()


check_cases(remove_parenthetical, [
    ("example (.com)", "example"),
    ("w3resource", "w3resource"),
    ("github (.com)", "github"),
    ("stackoverflow (.com)", "stackoverflow"),
    ("before(outer(inner) text)after(extra)", "beforeafter"),
    (" x() ", "x"),
])

'example (.com)' → 'example'
'w3resource' → 'w3resource'
'github (.com)' → 'github'
'stackoverflow (.com)' → 'stackoverflow'
'before(outer(inner) text)after(extra)' → 'beforeafter'
' x() ' → 'x'


## 二、从网页检索气候数据

### 1. 读取并观察 Madison 气候表

使用 `urllib.request` 获取原始 HTML，再以 `html.parser` 解析。查看 Madison 页面中标题以 `Climate data for` 开头的 `table`，统计其直接子标签和表格行。各网页的实际访问 URL 和日期均由运行输出记录。

In [15]:
from datetime import datetime, timedelta, timezone
from urllib.request import Request, urlopen

from bs4 import BeautifulSoup, Tag


MADISON_URL = "https://en.wikipedia.org/wiki/Madison,_Wisconsin"
HARBIN_URL = "https://en.wikipedia.org/wiki/Harbin"
LOS_ANGELES_URL = "https://en.wikipedia.org/wiki/Los_Angeles"
WIKIPEDIA_USER_AGENT = "PythonCoursework/1.0 (educational climate-table exercise)"
SHANGHAI_TIME = timezone(timedelta(hours=8))


def fetch_html(url):
    request = Request(url, headers={"User-Agent": WIKIPEDIA_USER_AGENT})
    with urlopen(request, timeout=20) as response:
        return response.read()


# 用此辅助函数统一处理换行、不换行空格和内联标签。
def normalized_text(tag):
    return " ".join(tag.get_text(" ", strip=True).split())


def table_rows(table):
    for row in table.find_all("tr"):
        if row.find_parent("table") is table:
            yield row


madison_html = fetch_html(MADISON_URL)
madison_page = BeautifulSoup(madison_html, "html.parser")
madison_heading = madison_page.find(
    "th", string=re.compile(r"^\s*Climate data for")
)
# 当前 Madison 标题包含链接和脚注，文本可能分布在多个子节点。
if madison_heading is None:
    madison_heading = next(
        header for header in madison_page.find_all("th")
        if normalized_text(header).startswith("Climate data for")
    )
madison_table = madison_heading.find_parent("table")
assert isinstance(madison_table, Tag) and madison_table.name == "table"
direct_children = madison_table.find_all(recursive=False)
madison_rows = list(table_rows(madison_table))
print("访问 URL", MADISON_URL)
print("访问日期", datetime.now(SHANGHAI_TIME).date().isoformat())
print("标题", normalized_text(madison_heading))
print("直接子标签数量", len(direct_children))
print("直接子标签名", [tag.name for tag in direct_children])
print("表内 tr 数量", len(madison_rows))
print("第一行标签名", [cell.name for cell in madison_rows[0].find_all(recursive=False)])
print("第二行首个单元格", normalized_text(madison_rows[1].find(["th", "td"])))

访问 URL https://en.wikipedia.org/wiki/Madison,_Wisconsin
访问日期 2026-11-25
标题 Climate data for Madison, Wisconsin ( Dane County Regional Airport ), 1991–2020 normals, [ a ] extremes 1869–present [ b ]
直接子标签数量 1
直接子标签名 ['tbody']
表内 tr 数量 18
第一行标签名 ['th']
第二行首个单元格 Month


本次访问时，Madison 表格的直接子标签只有一个 `tbody`。`tr` 位于 `tbody` 内，首行是标题 `th`，第二行是以 `Month` 开头的月份表头。数据行由行首 `th` 与后面的数据 `td` 组成，表末的数据来源占整行。

### 2. 根据 URL 查找气候表

依次检查每张表的 `caption`、首行 `th`（含内嵌的标题标签）和相邻标题。洛杉矶首行 `th` 包含导航文字以及单独的标题 `div`。找到第一张以 `Climate data for` 为标题前缀的表时返回 `Tag`。下载异常由 `urlopen` 抛出，网页成功读取且没有气候表时返回 `None`。

In [16]:
CLIMATE_TITLE = re.compile(r"^Climate data for\b", re.IGNORECASE)
NEARBY_HEADING_TAGS = {"h2", "h3", "h4", "h5", "h6", "p", "div"}


def climate_table_title(table):
    candidates = []
    caption = table.find("caption", recursive=False)
    if caption is not None:
        candidates.append(caption)

    first_row = next(table_rows(table), None)
    if first_row is not None:
        for header in first_row.find_all("th", recursive=False):
            candidates.append(header)
            candidates.extend(header.find_all(["div", "span", "strong"]))

    previous = table.find_previous_sibling()
    if previous is None and table.parent.name == "div":
        previous = table.parent.find_previous_sibling()
    if (previous is not None and previous.name in NEARBY_HEADING_TAGS
            and previous.find("table") is None):
        candidates.append(previous)

    for candidate in candidates:
        title = normalized_text(candidate)
        if CLIMATE_TITLE.match(title):
            return title
    return None


def retrieve_climate_table(s):
    """读取 URL s，返回第一张气候数据表的 Tag，或在未找到时返回 None。"""
    page = BeautifulSoup(fetch_html(s), "html.parser")
    for table in page.find_all("table"):
        if climate_table_title(table) is not None:
            return table
    return None

### 3. 提取气候数据表的行名

数据行的首格是 `th`，同行后面有数据 `td`。如果该 `th` 声明了 `scope`，只接受 `row`。跳过 `Month`、标题和 `Source` 行，并保留原始数据行顺序。

In [17]:
NON_DATA_ROW = re.compile(r"^(?:months?$|climate data for\b|sources?\b)", re.IGNORECASE)


def list_climate_table_row_names(s):
    """从 URL s 对应的气候表提取纯文本行名。"""
    table = retrieve_climate_table(s)
    if table is None:
        return None

    names = []
    for row in table_rows(table):
        cells = row.find_all(["th", "td"], recursive=False)
        if not cells or not any(cell.name == "td" for cell in cells[1:]):
            continue
        first = cells[0]
        if first.name != "th" or first.get("scope", "row").lower() != "row":
            continue
        name = normalized_text(first)
        if name and not NON_DATA_ROW.match(name):
            names.append(name)
    return names

### 测试不同的 HTML 结构与异常

用小型 HTML 样例检查 `caption`、表外标题、缺少 `scope`、列标题和来源行。另用模拟响应确认成功读到无表页面、HTTP 错误和网络错误时的行为。

In [18]:
from unittest.mock import patch
from urllib.error import HTTPError, URLError


test_url = "https://example.test/city"
sample_rows = """
<thead><tr><th scope="row">Month</th><td>Jan</td><td>Feb</td></tr></thead>
<tbody>
  <tr><th scope="col">Period</th><td>1</td><td>2</td></tr>
  <tr><th>  Daily <em>mean</em> °C </th><td>12</td><td>15</td></tr>
  <tr><th scope="row"> Rainfall&nbsp; mm </th><td>2</td><td>3</td></tr>
  <tr><th>Source 1:</th><td colspan="2">Sample data</td></tr>
</tbody>
"""
caption_sample = (
    "<table><tr><th>Population</th><td>100</td></tr></table>"
    "<table id='weather'><caption>Climate\n data <b>for</b> Sample City</caption>"
    f"{sample_rows}</table>"
)
heading_sample = f"<h3>Climate data for Sample City</h3><div><table>{sample_rows}</table></div>"
for sample in (caption_sample, heading_sample):
    with patch("__main__.fetch_html", return_value=sample):
        assert isinstance(retrieve_climate_table(test_url), Tag)
        assert list_climate_table_row_names(test_url) == ["Daily mean °C", "Rainfall mm"]
print("已通过：caption、相邻标题、无 scope 的行名和非数据行过滤")

with patch("__main__.fetch_html", return_value="<table><tr><th>Population</th></tr></table>"):
    assert retrieve_climate_table(test_url) is None
    assert list_climate_table_row_names(test_url) is None
print("已通过：成功读取无气候表的网页时返回 None")

for error in (
    HTTPError(test_url, 403, "Forbidden", {}, None),
    HTTPError(test_url, 429, "Too Many Requests", {}, None),
    URLError("network unavailable"),
):
    with patch("__main__.urlopen", side_effect=error):
        for function in (retrieve_climate_table, list_climate_table_row_names):
            try:
                function(test_url)
            except URLError as exc:
                assert exc is error
            else:
                raise AssertionError(f"{function.__name__} 应传出 {error!r}")
    print("已通过：", str(error))

已通过：caption、相邻标题、无 scope 的行名和非数据行过滤
已通过：成功读取无气候表的网页时返回 None
已通过： HTTP Error 403: Forbidden
已通过： HTTP Error 429: Too Many Requests
已通过： <urlopen error network unavailable>


### 在多个城市页面实测

Madison 和哈尔滨的标题文字直接位于 `th` 中。洛杉矶的标题则位于 `th` 内的 `div`，同一个 `th` 还有导航文字。测试检查表格类型、标题、预期的数据行和清理后的字符串行名。

In [19]:
city_tests = [
    ("Madison", MADISON_URL, "Record high °F (°C)"),
    ("哈尔滨", HARBIN_URL, "Record high °C (°F)"),
    ("洛杉矶", LOS_ANGELES_URL, "Record high °F (°C)"),
]

for city, url, expected_row in city_tests:
    table = retrieve_climate_table(url)
    assert isinstance(table, Tag) and table.name == "table", city
    title = climate_table_title(table)
    assert title and CLIMATE_TITLE.match(title), city
    if url == LOS_ANGELES_URL:
        first_header = next(table_rows(table)).find("th")
        assert not CLIMATE_TITLE.match(normalized_text(first_header))
        assert any(
            CLIMATE_TITLE.match(normalized_text(tag))
            for tag in first_header.find_all("div")
        )

    names = list_climate_table_row_names(url)
    assert names and expected_row in names, (city, names)
    assert all(
        isinstance(name, str) and name == name.strip() and not NON_DATA_ROW.match(name)
        for name in names
    )
    print(f"{city} | 访问日期 {datetime.now(SHANGHAI_TIME).date().isoformat()} | {url}")
    print("表格标题", title)
    print(f"共 {len(names)} 个数据行名")
    print(*names, sep="\n")
    print()

Madison | 访问日期 2026-11-25 | https://en.wikipedia.org/wiki/Madison,_Wisconsin
表格标题 Climate data for Madison, Wisconsin ( Dane County Regional Airport ), 1991–2020 normals, [ a ] extremes 1869–present [ b ]
共 15 个数据行名
Record high °F (°C)
Mean maximum °F (°C)
Mean daily maximum °F (°C)
Daily mean °F (°C)
Mean daily minimum °F (°C)
Mean minimum °F (°C)
Record low °F (°C)
Average precipitation inches (mm)
Average snowfall inches (cm)
Average extreme snow depth inches (cm)
Average precipitation days (≥ 0.01 in)
Average snowy days (≥ 0.1 in)
Average relative humidity (%)
Mean monthly sunshine hours
Percentage possible sunshine



哈尔滨 | 访问日期 2026-11-25 | https://en.wikipedia.org/wiki/Harbin
表格标题 Climate data for Harbin, elevation 118 m (387 ft), (1991–2020 normals, extremes 1951–present)
共 14 个数据行名
Record high °C (°F)
Mean maximum °C (°F)
Mean daily maximum °C (°F)
Daily mean °C (°F)
Mean daily minimum °C (°F)
Mean minimum °C (°F)
Record low °C (°F)
Average precipitation mm (inches)
Average precipitation days (≥ 0.1 mm)
Average snowy days
Average relative humidity (%)
Mean monthly sunshine hours
Percentage possible sunshine
Average ultraviolet index



洛杉矶 | 访问日期 2026-11-25 | https://en.wikipedia.org/wiki/Los_Angeles
表格标题 Climate data for Los Angeles ( Dodger Stadium , Downtown ), 1991–2020 normals, extremes 1877–present
共 12 个数据行名
Record high °F (°C)
Mean maximum °F (°C)
Mean daily maximum °F (°C)
Daily mean °F (°C)
Mean daily minimum °F (°C)
Mean minimum °F (°C)
Record low °F (°C)
Average precipitation inches (mm)
Average precipitation days (≥ 0.01 in)
Mean monthly sunshine hours
Percentage possible sunshine
Average ultraviolet index

